# ROLL A

In [21]:
with open('.env', 'w') as f:
    f.write('SUPABASE_URL=https://hydpsemnoyqtlktppbqo.supabase.co\n')
    f.write('SUPABASE_KEY=sb_publishable_TrAWxMJ8R9htxZv6yfdAvA_rKWQrPYZ\n')
print("File .env created!")

File .env created!


In [22]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Successful connection!")

Successful connection!


In [23]:
def fetch_all_rows(table, filters=None):
    all_data = []
    page_size = 1000
    offset = 0
    while True:
        query = supabase.table(table).select('*').range(offset, offset + page_size - 1)
        if filters:
            for method, column, value in filters:
                query = getattr(query, method)(column, value)
        response = query.execute()
        if not response.data:
            break
        all_data.extend(response.data)
        if len(response.data) < page_size:
            break
        offset += page_size
    return all_data

def fetch_sales(start_date=None, end_date=None):
    try:
        filters = []
        if start_date:
            filters.append(("gte", "sale_date", start_date))
        if end_date:
            filters.append(("lte", "sale_date", end_date))
        data = fetch_all_rows('sales', filters)
        df = pd.DataFrame(data)
        print(f"[fetch_sales] Laaditud {len(df)} rida.")
        return df
    except Exception as e:
        print(f"[fetch_sales] Viga: {e}")
        return pd.DataFrame()

df_sales = fetch_sales(start_date="2023-01-01", end_date="2024-12-31")
print(df_sales.head())

[fetch_sales] Laaditud 9411 rida.
   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3   4        4  INV-202301-00004  2023-01-02T00:00:00       4677.0   
4   5        5  INV-202301-00005  2023-01-13T00:00:00       2390.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method  
0          kaart  
1      järelmaks  
2      järelmaks  
3       sularaha  

In [24]:
def fetch_customers():
    try:
        data = fetch_all_rows('customers')
        df = pd.DataFrame(data)
        print(f"[fetch_customers] Laaditud {len(df)} rida.")
        return df
    except Exception as e:
        print(f"[fetch_customers] Viga: {e}")
        return pd.DataFrame()

df_customers = fetch_customers()
print(df_customers.head())

[fetch_customers] Laaditud 3150 rida.
   customer_id first_name last_name                   email           phone  \
0         2001        Eha       Aas        eha.aas@telia.ee  +372 8713 1455   
1         2002      Aivar      Kõiv  aivar.koiv@outlook.com  +372 8943 8684   
2         2003      Maris    Rebane   maris.rebane@telia.ee  +372 5918 5726   
3         2004       Jaak    Talvik     jaak.talvik@mail.ee  +372 8554 4232   
4         2005      Raivo    Koppel  raivo.koppel@yahoo.com  +372 5298 4365   

       city registration_date loyalty_tier  birth_year  
0   Tallinn        2024-02-27          NaN        1973  
1  Haapsalu        2025-01-09       bronze        1988  
2     Tartu        2021-02-03          NaN        1999  
3   Tallinn        2023-11-12       silver        1974  
4   Tallinn        2023-05-22       bronze        2004  


In [25]:
def fetch_products():
    try:
        data = fetch_all_rows('products')
        df = pd.DataFrame(data)
        print(f"[fetch_products] Laaditud {len(df)} rida.")
        return df
    except Exception as e:
        print(f"[fetch_products] Viga: {e}")
        return pd.DataFrame()

df_products = fetch_products()
print(df_products.head())

[fetch_products] Laaditud 362 rida.
   product_id                      product_name       category subcategory  \
0        1001  Praktiline seemisnahkne tennised     Jalanõusid      tossud   
1        1002       Mugav linane linane kostüüm  Meeste_riided   ülikonnad   
2        1003       Elegantne orgaaniline kleit   Laste_riided     kleidid   
3        1004  Minimalistlik puuvillane tuunika  Naiste_riided     pluusid   
4        1005              Mugav tweed kardigan  Meeste_riided   kampsunid   

                  supplier  cost_price  retail_price eco_certified  created_at  
0          Soome Tehdas OY       99.21        157.49         False  2022-05-12  
1           Riia Stils SIA      173.56        274.78          True  2022-05-10  
2      Rakvere Tekstiil OÜ       31.83         52.04         False  2020-02-19  
3          Vilma Design OÜ       29.41         41.63         False  2022-12-26  
4  Nordic Fashion Group OÜ      135.70        198.29          True  2021-06-19  
